In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# =====================================================
# SYNTHETIC DATA GENERATION  ("X" vs "O", 8x8 binary images)
# =====================================================
# No external dataset needed: X and O shapes are drawn directly onto
# an 8x8 grid, with small random translations + random pixel noise
# added to each sample so the CNN sees genuine (if small) variation,
# not 200 copies of 2 fixed images.

IMG_SIZE = 8

def make_X(shift=(0, 0), noise_pixels=2, rng=np.random):
    grid = np.zeros((IMG_SIZE, IMG_SIZE))
    for k in range(IMG_SIZE):
        for (i, j) in [(k, k), (k, IMG_SIZE - 1 - k)]:
            ii, jj = i + shift[0], j + shift[1]
            if 0 <= ii < IMG_SIZE and 0 <= jj < IMG_SIZE:
                grid[ii, jj] = 1.0
    for _ in range(noise_pixels):
        ii, jj = rng.randint(0, IMG_SIZE), rng.randint(0, IMG_SIZE)
        grid[ii, jj] = 1.0 - grid[ii, jj]   # flip a random pixel
    return grid

def make_O(shift=(0, 0), noise_pixels=2, rng=np.random):
    grid = np.zeros((IMG_SIZE, IMG_SIZE))
    center = (IMG_SIZE - 1) / 2.0
    radius = IMG_SIZE / 2.0 - 1.0
    for i in range(IMG_SIZE):
        for j in range(IMG_SIZE):
            d = np.sqrt((i - center) ** 2 + (j - center) ** 2)
            if abs(d - radius) < 0.9:
                ii, jj = i + shift[0], j + shift[1]
                if 0 <= ii < IMG_SIZE and 0 <= jj < IMG_SIZE:
                    grid[ii, jj] = 1.0
    for _ in range(noise_pixels):
        ii, jj = rng.randint(0, IMG_SIZE), rng.randint(0, IMG_SIZE)
        grid[ii, jj] = 1.0 - grid[ii, jj]
    return grid

rng = np.random.RandomState(42)
N_PER_CLASS = 150

images, labels = [], []

for _ in range(N_PER_CLASS):
    shift = (rng.randint(-1, 2), rng.randint(-1, 2))   # -1, 0, or +1
    images.append(make_X(shift=shift, noise_pixels=2, rng=rng))
    labels.append(1)   # X = 1

for _ in range(N_PER_CLASS):
    shift = (rng.randint(-1, 2), rng.randint(-1, 2))
    images.append(make_O(shift=shift, noise_pixels=2, rng=rng))
    labels.append(0)   # O = 0

images = np.array(images)                       # (N, 8, 8)
labels = np.array(labels).reshape(-1, 1)         # (N, 1)

# add a channel dimension: (N, 1, 8, 8)  -> C=1 (grayscale)
images = images[:, np.newaxis, :, :]

print("images shape:", images.shape)
print("labels shape:", labels.shape)

# =====================================================
# TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)
print(X_train.shape)
print(y_train.shape)

# Pixels are already binary (0/1), so no StandardScaler is needed here
# (unlike the temperature data used in the MLP/RNN/LSTM projects).

# =====================================================
# CNN PARAMETERS
# =====================================================

num_filters  = 4     # F
in_channels  = 1      # C
kernel_size  = 3      # K
conv_stride  = 1

pool_size    = 2
pool_stride  = 2

conv_out_dim  = IMG_SIZE - kernel_size + 1         # 8 - 3 + 1 = 6
pooled_dim    = conv_out_dim // pool_stride         # 6 / 2     = 3
flatten_size  = num_filters * pooled_dim * pooled_dim  # 4*3*3 = 36

hidden_size  = 16     # dense hidden layer
output_size  = 1      # binary classification

learning_rate = 0.01
epochs = 300

# =====================================================
# WEIGHTS
# =====================================================

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Conv layer: F filters, each (C, K, K)
Wc = np.random.randn(num_filters, in_channels, kernel_size, kernel_size) * 0.1
bc = np.zeros((num_filters, 1))

# Dense layer 1: flatten -> hidden
Wd1 = np.random.randn(hidden_size, flatten_size) * 0.1
bd1 = np.zeros((hidden_size, 1))

# Dense layer 2 (output): hidden -> 1  (linear, sigmoid applied after)
Wd2 = np.random.randn(output_size, hidden_size) * 0.1
bd2 = np.zeros((output_size, 1))

# =====================================================
# FORWARD / BACKWARD HELPERS
# =====================================================

def conv_forward(x, W, b):
    """x: (C,H,W)  W: (F,C,K,K)  b: (F,1)  ->  z: (F, out_h, out_w)"""
    F, C, K, _ = W.shape
    out_h = x.shape[1] - K + 1
    out_w = x.shape[2] - K + 1
    z = np.zeros((F, out_h, out_w))
    for f in range(F):
        for i in range(out_h):
            for j in range(out_w):
                patch = x[:, i:i+K, j:j+K]
                z[f, i, j] = np.sum(W[f] * patch) + b[f, 0]
    return z

def maxpool_forward(a, pool_size, pool_stride):
    """a: (F,H,W) -> pooled: (F, H//p, W//p), argmax positions for backprop"""
    F, H, W = a.shape
    ph = H // pool_stride
    pw = W // pool_stride
    pooled = np.zeros((F, ph, pw))
    argmax = np.zeros((F, ph, pw, 2), dtype=int)  # store (row, col) of max
    for f in range(F):
        for i in range(ph):
            for j in range(pw):
                window = a[f, i*pool_stride:i*pool_stride+pool_size,
                              j*pool_stride:j*pool_stride+pool_size]
                idx = np.unravel_index(np.argmax(window), window.shape)
                argmax[f, i, j] = idx
                pooled[f, i, j] = window[idx]
    return pooled, argmax

# =====================================================
# TRAINING
# =====================================================

loss_history = []

for epoch in range(epochs):

    total_loss = 0

    for n in range(len(X_train)):

        x = X_train[n]                    # (1, 8, 8)
        target = y_train[n, 0]            # scalar 0 or 1

        # ==========================================
        # FORWARD PASS
        # ==========================================

        z_conv = conv_forward(x, Wc, bc)          # (F, 6, 6)
        a_conv = np.maximum(0, z_conv)             # ReLU

        pooled, argmax = maxpool_forward(a_conv, pool_size, pool_stride)  # (F,3,3)

        flat = pooled.reshape(-1, 1)               # (36, 1)

        z1 = Wd1 @ flat + bd1
        h1 = np.tanh(z1)                           # (16, 1)

        z2 = Wd2 @ h1 + bd2
        y_hat = sigmoid(z2)                        # (1, 1) probability of "X"

        eps = 1e-9
        loss = -(target * np.log(y_hat[0,0] + eps) +
                 (1 - target) * np.log(1 - y_hat[0,0] + eps))
        total_loss += loss

        # ==========================================
        # OUTPUT LAYER GRADIENT (sigmoid + BCE combine nicely)
        # ==========================================

        dz2 = y_hat - target             # (1,1)

        dWd2 = dz2 @ h1.T
        dbd2 = dz2

        # ==========================================
        # DENSE HIDDEN LAYER GRADIENT
        # ==========================================

        dh1 = Wd2.T @ dz2
        dz1 = dh1 * (1 - h1 ** 2)         # tanh'(z1) = 1 - h1^2

        dWd1 = dz1 @ flat.T
        dbd1 = dz1

        # ==========================================
        # FLATTEN -> POOLED GRADIENT
        # ==========================================

        dflat  = Wd1.T @ dz1
        dpooled = dflat.reshape(pooled.shape)     # (F, 3, 3)

        # ==========================================
        # MAXPOOL BACKWARD (route gradient to the max location)
        # ==========================================

        da_conv = np.zeros_like(a_conv)
        F_, ph, pw = dpooled.shape
        for f in range(F_):
            for i in range(ph):
                for j in range(pw):
                    di, dj = argmax[f, i, j]
                    da_conv[f, i*pool_stride+di, j*pool_stride+dj] += dpooled[f, i, j]

        # ==========================================
        # RELU BACKWARD
        # ==========================================

        dz_conv = da_conv * (z_conv > 0)

        # ==========================================
        # CONV LAYER GRADIENT
        # ==========================================

        dWc = np.zeros_like(Wc)
        dbc = np.zeros_like(bc)
        K = kernel_size
        for f in range(num_filters):
            for i in range(conv_out_dim):
                for j in range(conv_out_dim):
                    patch = x[:, i:i+K, j:j+K]
                    dWc[f] += dz_conv[f, i, j] * patch
            dbc[f, 0] = np.sum(dz_conv[f])

        # ==========================================
        # WEIGHT UPDATE
        # ==========================================

        Wc -= learning_rate * dWc
        bc -= learning_rate * dbc

        Wd1 -= learning_rate * dWd1
        bd1 -= learning_rate * dbd1

        Wd2 -= learning_rate * dWd2
        bd2 -= learning_rate * dbd2

    loss_history.append(total_loss / len(X_train))

    if epoch % 20 == 0:
        print(f"Epoch {epoch:4d}   Loss = {total_loss / len(X_train):.6f}")

# =====================================================
# TESTING
# =====================================================

predictions = []

for n in range(len(X_test)):

    x = X_test[n]

    z_conv = conv_forward(x, Wc, bc)
    a_conv = np.maximum(0, z_conv)

    pooled, _ = maxpool_forward(a_conv, pool_size, pool_stride)
    flat = pooled.reshape(-1, 1)

    z1 = Wd1 @ flat + bd1
    h1 = np.tanh(z1)

    z2 = Wd2 @ h1 + bd2
    y_hat = sigmoid(z2)

    predictions.append(1 if y_hat[0, 0] >= 0.5 else 0)

predictions = np.array(predictions).reshape(-1, 1)

# =====================================================
# EVALUATION
# =====================================================

acc = accuracy_score(y_test, predictions)
cm  = confusion_matrix(y_test, predictions)
print(f"Test Accuracy : {acc*100:.2f}%")
print("Confusion matrix (rows=actual, cols=predicted; 0=O, 1=X):")
print(cm)

# =====================================================
# PLOTS
# =====================================================

plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Avg. Binary Cross-Entropy Loss")
plt.title("Training Loss (Scratch CNN, X vs O)")
plt.grid(True)
plt.show()

# show a few test predictions
n_show = min(8, len(X_test))
fig, axes = plt.subplots(1, n_show, figsize=(2*n_show, 2.5))
for idx in range(n_show):
    ax = axes[idx]
    ax.imshow(X_test[idx, 0], cmap="gray_r")
    true_label = "X" if y_test[idx, 0] == 1 else "O"
    pred_label = "X" if predictions[idx, 0] == 1 else "O"
    ax.set_title(f"T:{true_label} P:{pred_label}", fontsize=10)
    ax.axis("off")
plt.suptitle("Sample Test Predictions")
plt.tight_layout()
plt.show()